# Verify MOT -> YOLO conversion

Two checks on `dataset/` (produced by `scripts/convert_mot_to_yolo.py`):

1. **Visual** - draw the YOLO boxes on random sample images to confirm the class map and
   coordinate conversion (MOT top-left px -> YOLO normalized center) are correct.
2. **Completeness** - compare image counts and box counts against the raw MOT source directory
   source so nothing silently got dropped during conversion.

In [ ]:
import random
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import yaml
from PIL import Image

ROOT = Path.cwd().parent
DATASET = ROOT / "dataset"
RAW_DATA = ROOT / "raw_data"

data_cfg = yaml.safe_load((DATASET / "data.yaml").read_text())
CLASS_NAMES = data_cfg["names"]
print(CLASS_NAMES)

## 1. Visual check - draw boxes on random samples

In [ ]:
COLORS = {0: "red", 1: "orange", 2: "lime", 3: "yellow", 4: "cyan"}  # ball, goalkeeper, player, referee, other


def load_boxes(label_path: Path):
    if not label_path.exists() or label_path.stat().st_size == 0:
        return []
    boxes = []
    for line in label_path.read_text().splitlines():
        cid, cx, cy, w, h = line.split()
        boxes.append((int(cid), float(cx), float(cy), float(w), float(h)))
    return boxes


def plot_sample(ax, img_path: Path, label_path: Path):
    img = Image.open(img_path)
    iw, ih = img.size
    ax.imshow(img)
    for cid, cx, cy, w, h in load_boxes(label_path):
        x = (cx - w / 2) * iw
        y = (cy - h / 2) * ih
        rect = patches.Rectangle(
            (x, y), w * iw, h * ih, linewidth=1.5, edgecolor=COLORS[cid], facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(x, max(y - 4, 0), CLASS_NAMES[cid], color=COLORS[cid], fontsize=7, weight="bold")
    ax.set_title(img_path.stem, fontsize=8)
    ax.axis("off")


def show_random_grid(split: str, n: int = 9, seed: int = 0):
    images_dir = DATASET / "images" / split
    labels_dir = DATASET / "labels" / split
    all_images = sorted(images_dir.glob("*.jpg"))
    sample = random.Random(seed).sample(all_images, n)

    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    for ax, img_path in zip(axes.flat, sample):
        label_path = labels_dir / f"{img_path.stem}.txt"
        plot_sample(ax, img_path, label_path)
    for ax in axes.flat[len(sample):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
show_random_grid("train", n=9, seed=0)

In [ ]:
# re-run with a different seed / split to spot-check more samples, e.g.:
show_random_grid("val", n=9, seed=1)
show_random_grid("test", n=9, seed=2)

## 2. Completeness check - nothing lost between the raw source and dataset/

In [ ]:
# image counts: every source frame should have exactly one YOLO image (symlink) + one label file
src_train_imgs = sum(len(list((p / "img1").glob("*.jpg"))) for p in (RAW_DATA / "train").iterdir())
src_test_imgs = sum(len(list((p / "img1").glob("*.jpg"))) for p in (RAW_DATA / "test").iterdir())

yolo_train_imgs = len(list((DATASET / "images" / "train").glob("*.jpg")))
yolo_val_imgs = len(list((DATASET / "images" / "val").glob("*.jpg")))
yolo_test_imgs = len(list((DATASET / "images" / "test").glob("*.jpg")))

print(f"raw    train frames           : {src_train_imgs:,}")
print(f"yolo   train+val frames       : {yolo_train_imgs + yolo_val_imgs:,}")
assert src_train_imgs == yolo_train_imgs + yolo_val_imgs, "train/val frame count mismatch!"

print(f"raw    test frames            : {src_test_imgs:,}")
print(f"yolo   test frames            : {yolo_test_imgs:,}")
assert src_test_imgs == yolo_test_imgs, "test frame count mismatch!"

print("OK: image counts match.")

In [ ]:
# box counts: every gt.txt row should map to exactly one YOLO label line (all roles are
# covered by ROLE_MAP in convert_mot_to_yolo.py, so nothing should be silently dropped)
def count_gt_rows(split_dir: Path) -> int:
    total = 0
    for seq_dir in split_dir.iterdir():
        if not seq_dir.is_dir():
            continue
        gt = seq_dir / "gt" / "gt.txt"
        total += sum(1 for line in gt.read_text().splitlines() if line.strip())
    return total


def count_yolo_boxes(labels_dir: Path) -> int:
    total = 0
    for lbl in labels_dir.glob("*.txt"):
        total += sum(1 for line in lbl.read_text().splitlines() if line.strip())
    return total


src_train_boxes = count_gt_rows(RAW_DATA / "train")
src_test_boxes = count_gt_rows(RAW_DATA / "test")
yolo_train_boxes = count_yolo_boxes(DATASET / "labels" / "train") + count_yolo_boxes(DATASET / "labels" / "val")
yolo_test_boxes = count_yolo_boxes(DATASET / "labels" / "test")

print(f"raw    train gt.txt rows      : {src_train_boxes:,}")
print(f"yolo   train+val boxes        : {yolo_train_boxes:,}")
assert src_train_boxes == yolo_train_boxes, "train/val box count mismatch!"

print(f"raw    test gt.txt rows       : {src_test_boxes:,}")
print(f"yolo   test boxes             : {yolo_test_boxes:,}")
assert src_test_boxes == yolo_test_boxes, "test box count mismatch!"

print("OK: box counts match.")

In [ ]:
# broken symlinks: images/*.jpg should all resolve to a real file on disk
broken = []
for split in ("train", "val", "test"):
    for img in (DATASET / "images" / split).glob("*.jpg"):
        if not img.resolve().exists():
            broken.append(img)

print(f"broken symlinks: {len(broken)}")
for b in broken[:10]:
    print(" ", b)

In [ ]:
from ultralytics import YOLO

YOLO("yolo26n.pt")